Public analyses of this dataset may include post-origination columns. This appendix shows that this leads to discrimination that is largely due to leakage. The model in 03_modelling.ipynb should be read against this.

In [8]:
import duckdb, lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys; sys.path.append("../src")
from lending_pd.config import LEAKY_COLUMNS

con = duckdb.connect("../data/processed/lending.duckdb", read_only=True)
df = con.execute("SELECT * FROM cohort").df()

leaky_numeric = [c for c in LEAKY_COLUMNS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
X, y = df[leaky_numeric], df["default_flag"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clfs = lgb.LGBMClassifier(n_estimators=200, random_state=42)
clfs.fit(X_tr, y_tr)
print("AUC on leaky features:", roc_auc_score(y_te, clfs.predict_proba(X_te)[:, 1]))

[LightGBM] [Info] Number of positive: 55672, number of negative: 344983
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005537 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3188
[LightGBM] [Info] Number of data points in the train set: 400655, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.138952 -> initscore=-1.824018
[LightGBM] [Info] Start training from score -1.824018
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

AUC = 1 is perfect discrimination. Demonstrates included columns contain restatements of loan amount outcomes. Positive test for post-origination status / leakage from these (excluded) columns.

In [11]:
imp = pd.Series(clfs.feature_importances_, index=leaky_numeric).sort_values(ascending=False)
print(imp.head(8))

total_pymnt             407
default_flag            176
total_rec_int            90
total_rec_prncp          85
total_rec_late_fee       82
total_pymnt_inv          57
last_pymnt_amnt          54
last_fico_range_high     23
dtype: int32


Leaky columns/features by order of decreasing influence on exhibit outcome.